In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
!pip -q install "transformers==5.17.0"


In [ ]:
import glob, os
hits = glob.glob("/kaggle/input/**/terrain_relief_kaggle.py", recursive=True)
assert len(hits) == 1, hits          # 0 = dataset not attached, 2 = attached twice
BUNDLE = os.path.dirname(hits[0])
OUT = "/kaggle/working/terrain_ctrl"
LOG = "/kaggle/working/terrain_ctrl_run_log.txt"
os.makedirs(OUT, exist_ok=True)
print("BUNDLE =", BUNDLE); print(os.listdir(BUNDLE))


In [ ]:
import shutil
prev = [p for p in glob.glob("/kaggle/input/**/terrain_ctrl", recursive=True) if os.path.isdir(p)]
assert len(prev) <= 1, prev
if prev:
    for f in os.listdir(prev[0]):
        src, dst = os.path.join(prev[0], f), os.path.join(OUT, f)
        if f.endswith(".tmp"):
            continue                  # a half-written checkpoint: drop it
        (shutil.copytree if os.path.isdir(src) else shutil.copy)(src, dst)
    print("resumed from", prev[0], sorted(os.listdir(OUT)))
else:
    print("no previous output found: fresh start")


In [ ]:
import subprocess, time, sys
STALL_MIN, MAX_RESTARTS = 15, 5
cmd = [sys.executable, "-u", f"{BUNDLE}/terrain_relief_kaggle.py", "run", "--data", BUNDLE, "--out", OUT]
for attempt in range(MAX_RESTARTS + 1):
    print(f"=== launch {attempt} ===", flush=True)
    with open(LOG, "a") as lf:
        lf.write(f"=== launch {attempt} {time.ctime()} ===\n")
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        os.set_blocking(p.stdout.fileno(), False)
        last = time.time(); done = False
        while True:
            line = p.stdout.readline()
            if line:
                print(line, end="", flush=True); lf.write(line); lf.flush(); last = time.time()
                done = done or line.strip().endswith("DONE")
                continue
            if p.poll() is not None:
                break
            if time.time() - last > STALL_MIN * 60:
                print(f"!!! no output for {STALL_MIN} min: killing and restarting", flush=True)
                lf.write("!!! stall: killed\n"); p.kill(); p.wait(); break
            time.sleep(2)
    if done and p.returncode == 0:
        print("=== finished ==="); break
    print(f"=== exited with code {p.returncode}; restarting in 30 s ===", flush=True); time.sleep(30)
else:
    raise RuntimeError("gave up after MAX_RESTARTS restarts; see LOG")


In [ ]:
print(sorted(os.listdir(OUT)))
print(open(f"{OUT}/raw_spearman/summary.json").read()[:1500])
